# MOR Position Delete File Lab

이 실험은 운영 테이블이 아니라 별도 실험 테이블 `mor_orders_lab`을 사용한다.

## 목표

1. MOR table 생성
2. 초기 DATA file 생성
3. `UPDATE` / `DELETE`로 position delete file 발생 여부 확인
4. `files.content` 분포 비교
5. `rewrite_position_delete_files` 실행 전후 비교

## 결과 해석 기준

Iceberg `files.content` 값은 다음처럼 해석한다.

| content | 의미 |
|---:|---|
| 0 | DATA file |
| 1 | POSITION DELETE file |
| 2 | EQUALITY DELETE file |

기대 흐름은 다음과 같다.

```text
insert 후        : DATA file만 존재
UPDATE 후        : POSITION DELETE file 생성
DELETE 후        : POSITION DELETE file 추가 생성 가능
rewrite 후       : delete file들이 더 적은 수의 delete file로 rewrite됨
```

In [1]:
import os

os.environ["PYSPARK_PYTHON"] = ".venv/bin/python"
os.environ["PYSPARK_DRIVER_PYTHON"] = ".venv/bin/python"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("mor-position-delete-lab")
    .config("spark.jars.packages", ",".join([
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.0",
        "org.apache.iceberg:iceberg-aws-bundle:1.7.0",
        "org.apache.hadoop:hadoop-aws:3.3.4",
    ]))
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.catalog.glue", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.glue.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
    .config("spark.sql.catalog.glue.warehouse", "s3://binance-iceberg-lake/warehouse")
    .config("spark.sql.catalog.glue.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")

    # JVM crash 회피용 안정성 옵션
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .config("spark.sql.iceberg.vectorization.enabled", "false")

    # 로컬 실험용
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "3g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

:: loading settings :: url = jar:file:/home/ubuntu/binance-iceberg-lakehouse/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ubuntu/.ivy2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-96f91b28-fa62-4f2c-9216-694463bfa3a0;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.7.0 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.7.0 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 295ms :: artifacts dl 15ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-aws-bundle;1.7.0 from cen

In [3]:
from pathlib import Path
import pandas as pd
import time

# NOTE: 기존 노트북의 결과 저장 경로를 유지한다.
RESULT_DIR = Path("results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

LAB_TABLE = "glue.binance_lakehouse.mor_orders_lab"
LAB_TABLE_PROC = "binance_lakehouse.mor_orders_lab"


def sql_df(query: str):
    """Spark SQL 실행 후 Spark DataFrame 반환."""
    return spark.sql(query)


def show_pd(query: str) -> pd.DataFrame:
    """Spark SQL 실행 후 작은 결과를 pandas DataFrame으로 반환."""
    return spark.sql(query).toPandas()


def timed_sql(query: str):
    """Spark SQL 실행 시간 측정."""
    start = time.time()
    df = spark.sql(query)
    rows = df.collect()
    elapsed = time.time() - start
    return rows, elapsed


def save_markdown(df: pd.DataFrame, path: str, title: str):
    """pandas DataFrame을 Markdown 파일로 저장."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)

    with out.open("w", encoding="utf-8") as f:
        f.write(f"# {title}")
        f.write(df.to_markdown(index=False))
        f.write("\n")


def content_distribution(table: str = LAB_TABLE) -> pd.DataFrame:
    """Iceberg files metadata table의 content 분포 조회."""
    return show_pd(f"""
        SELECT
            content,
            CASE
                WHEN content = 0 THEN 'DATA'
                WHEN content = 1 THEN 'POSITION_DELETE'
                WHEN content = 2 THEN 'EQUALITY_DELETE'
                ELSE 'UNKNOWN'
            END AS content_type,
            COUNT(*) AS file_count
        FROM {table}.files
        GROUP BY content
        ORDER BY content
    """)


def file_stats(table: str = LAB_TABLE) -> pd.DataFrame:
    """Iceberg files metadata table의 파일 통계 조회."""
    return show_pd(f"""
        SELECT
            COUNT(*) AS file_count,
            ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 6) AS avg_file_size_mb,
            ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 6) AS total_size_mb,
            SUM(record_count) AS total_records
        FROM {table}.files
    """)


def manifest_count(table: str = LAB_TABLE) -> pd.DataFrame:
    """Iceberg manifests metadata table의 manifest 개수 조회."""
    return show_pd(f"""
        SELECT COUNT(*) AS manifest_count
        FROM {table}.manifests
    """)


def snapshots(table: str = LAB_TABLE) -> pd.DataFrame:
    """Iceberg snapshots metadata table 조회."""
    return show_pd(f"""
        SELECT
            committed_at,
            operation,
            summary
        FROM {table}.snapshots
        ORDER BY committed_at
    """)

## 1. Lab table 생성

운영 테이블을 직접 건드리지 않기 위해 별도 MOR 실험 테이블을 생성한다.  
매 실행마다 `DROP TABLE IF EXISTS`로 초기화한다.

In [4]:
spark.sql(f"DROP TABLE IF EXISTS {LAB_TABLE}")

spark.sql(f"""
CREATE TABLE {LAB_TABLE} (
    order_id       STRING,
    symbol         STRING,
    order_status   STRING,
    order_qty      DECIMAL(20, 8),
    filled_qty     DECIMAL(20, 8),
    updated_at     TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(updated_at))
TBLPROPERTIES (
    'format-version' = '2',
    'write.update.mode' = 'merge-on-read',
    'write.merge.mode' = 'merge-on-read',
    'write.delete.mode' = 'merge-on-read'
)
""")

DataFrame[]

In [5]:
show_pd(f"SHOW TBLPROPERTIES {LAB_TABLE}").query(
    "key in ['format-version', 'write.update.mode', 'write.merge.mode', 'write.delete.mode']"
)

,key,value
2,format-version,2
3,write.delete.mode,merge-on-read
4,write.merge.mode,merge-on-read
6,write.update.mode,merge-on-read


## 2. 초기 데이터 INSERT

초기 insert 직후에는 DATA file만 존재하는 것이 정상이다.

In [6]:
spark.sql(f"""
INSERT INTO {LAB_TABLE} VALUES
('O0001', 'BTCUSDT', 'NEW', CAST(0.10000000 AS DECIMAL(20,8)), CAST(0.00000000 AS DECIMAL(20,8)), TIMESTAMP '2026-05-07 13:00:00'),
('O0002', 'BTCUSDT', 'NEW', CAST(0.20000000 AS DECIMAL(20,8)), CAST(0.00000000 AS DECIMAL(20,8)), TIMESTAMP '2026-05-07 13:00:00'),
('O0003', 'BTCUSDT', 'NEW', CAST(0.30000000 AS DECIMAL(20,8)), CAST(0.00000000 AS DECIMAL(20,8)), TIMESTAMP '2026-05-07 13:00:00'),
('O0004', 'BTCUSDT', 'NEW', CAST(0.40000000 AS DECIMAL(20,8)), CAST(0.00000000 AS DECIMAL(20,8)), TIMESTAMP '2026-05-07 13:00:00'),
('O0005', 'BTCUSDT', 'NEW', CAST(0.50000000 AS DECIMAL(20,8)), CAST(0.00000000 AS DECIMAL(20,8)), TIMESTAMP '2026-05-07 13:00:00')
""")

before_content = content_distribution().assign(stage="before_update")
before_stats = file_stats().assign(stage="before_update")

before_content

,content,content_type,file_count,stage
0,0,DATA,1,before_update


## 3. UPDATE 실행

MOR table에서 기존 row를 update하면 기존 data row를 무효화하기 위해 position delete file이 생성될 수 있다.

In [7]:
spark.sql(f"""
UPDATE {LAB_TABLE}
SET
    order_status = 'FILLED',
    filled_qty = order_qty,
    updated_at = TIMESTAMP '2026-05-07 13:10:00'
WHERE order_id IN ('O0001', 'O0002', 'O0003')
""")

after_update_content = content_distribution().assign(stage="after_update")
after_update_stats = file_stats().assign(stage="after_update")

after_update_content

,content,content_type,file_count,stage
0,0,DATA,2,after_update
1,1,POSITION_DELETE,1,after_update


In [8]:
show_pd(f"""
SELECT order_id, order_status, order_qty, filled_qty, updated_at
FROM {LAB_TABLE}
ORDER BY order_id
""")

,order_id,order_status,order_qty,filled_qty,updated_at
0,O0001,FILLED,0.10000000,0.10000000,2026-05-07 13:10:00
1,O0002,FILLED,0.20000000,0.20000000,2026-05-07 13:10:00
2,O0003,FILLED,0.30000000,0.30000000,2026-05-07 13:10:00
3,O0004,NEW,0.40000000,0E-8,2026-05-07 13:00:00
4,O0005,NEW,0.50000000,0E-8,2026-05-07 13:00:00


## 4. DELETE 실행

DELETE도 MOR table에서는 position delete file을 추가로 만들 수 있다.

In [9]:
spark.sql(f"""
DELETE FROM {LAB_TABLE}
WHERE order_id = 'O0005'
""")

after_delete_content = content_distribution().assign(stage="after_delete")
after_delete_stats = file_stats().assign(stage="after_delete")

after_delete_content

,content,content_type,file_count,stage
0,0,DATA,2,after_delete
1,1,POSITION_DELETE,2,after_delete


In [10]:
show_pd(f"""
SELECT order_id, order_status, order_qty, filled_qty, updated_at
FROM {LAB_TABLE}
ORDER BY order_id
""")

,order_id,order_status,order_qty,filled_qty,updated_at
0,O0001,FILLED,0.10000000,0.10000000,2026-05-07 13:10:00
1,O0002,FILLED,0.20000000,0.20000000,2026-05-07 13:10:00
2,O0003,FILLED,0.30000000,0.30000000,2026-05-07 13:10:00
3,O0004,NEW,0.40000000,0E-8,2026-05-07 13:00:00


## 5. `rewrite_position_delete_files` 실행

`after_delete` 단계에서 position delete file이 2개가 되면, 이를 rewrite하여 더 적은 delete file로 합칠 수 있는지 확인한다.

In [11]:
rewrite_result = spark.sql(f"""
CALL glue.system.rewrite_position_delete_files(
  table => '{LAB_TABLE_PROC}',
  options => map('rewrite-all','true')
)
""").toPandas()

rewrite_result

,rewritten_delete_files_count,added_delete_files_count,rewritten_bytes_count,added_bytes_count
0,2,1,3509,1782


In [12]:
after_rewrite_content = content_distribution().assign(stage="after_rewrite_position_delete_files")
after_rewrite_stats = file_stats().assign(stage="after_rewrite_position_delete_files")

after_rewrite_content

,content,content_type,file_count,stage
0,0,DATA,2,after_rewrite_position_delete_files
1,1,POSITION_DELETE,1,after_rewrite_position_delete_files


## 6. 결과 저장

기존 저장 경로인 `results/mor_position_delete_lab.md`를 유지한다.

수정 사항:

- `after_delete` 단계도 최종 결과에 포함한다.
- `content_type` 컬럼을 추가해 `0`, `1` 숫자만 봐도 의미를 알 수 있게 한다.
- `rewrite_position_delete_files` 실행 결과를 별도 파일로 저장한다.
- 파일 통계도 함께 저장한다.

In [13]:
content_result = pd.concat(
    [
        before_content,
        after_update_content,
        after_delete_content,
        after_rewrite_content,
    ],
    ignore_index=True,
)

save_markdown(
    content_result,
    "results/mor_position_delete_lab.md",
    "MOR Position Delete File Lab Result"
)

content_result

,content,content_type,file_count,stage
0,0,DATA,1,before_update
1,0,DATA,2,after_update
2,1,POSITION_DELETE,1,after_update
3,0,DATA,2,after_delete
4,1,POSITION_DELETE,2,after_delete
5,0,DATA,2,after_rewrite_position_delete_files
6,1,POSITION_DELETE,1,after_rewrite_position_delete_files


In [14]:
stats_result = pd.concat(
    [
        before_stats,
        after_update_stats,
        after_delete_stats,
        after_rewrite_stats,
    ],
    ignore_index=True,
)

save_markdown(
    stats_result,
    "results/mor_position_delete_file_stats.md",
    "MOR Position Delete File Stats"
)

stats_result

,file_count,avg_file_size_mb,total_size_mb,total_records,stage
0,1,0.001838,0.001838,5,before_update
1,3,0.001804,0.005411,11,after_update
2,4,0.001765,0.007059,12,after_delete
3,3,0.001804,0.005412,12,after_rewrite_position_delete_files


In [15]:
save_markdown(
    rewrite_result,
    "results/mor_position_delete_rewrite_result.md",
    "Rewrite Position Delete Files Procedure Result"
)

rewrite_result

,rewritten_delete_files_count,added_delete_files_count,rewritten_bytes_count,added_bytes_count
0,2,1,3509,1782


## 7. 결론

이 실험에서 확인할 핵심 결과는 다음과 같다.

```text
before_update: DATA file만 존재

UPDATE 후:
- DATA file 증가
- POSITION DELETE file 생성

DELETE 후:
- POSITION DELETE file 추가 생성

rewrite_position_delete_files 후:
- 여러 position delete file이 더 적은 delete file로 rewrite됨
```

따라서 MOR table에서 row-level update/delete가 position delete file로 표현되고,
Iceberg maintenance procedure로 delete file을 재작성할 수 있음을 확인한다.